## 결정 트리 실습

### 데이터 불러오기

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# features.txt 파일에는 피처 이름 index와 피처명이 공백으로 분리되어 있음. 이를 DataFrame으로 로드

feature_name_df = pd.read_csv('./human_activity/features.txt', sep='\s+',
                                header=None, names=['column_index', 'column_name'])

# 피처명 index를 제거하고, 피처명만 리스트 객체로 생성한 뒤 샘플로 10개만 추출
feature_name = feature_name_df.iloc[:, 1].values.tolist()
print('전체 피처명에서 10개만 추출:', feature_name[:10])

전체 피처명에서 10개만 추출: ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z', 'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z', 'tBodyAcc-max()-X']


In [3]:
## 피처명 중복 확인

feature_dup_df = feature_name_df.groupby('column_name').count() # 이름이 같은 것끼리 모아서 중복된 것이 몇 개인지 확인
print(feature_dup_df[feature_dup_df['column_index'] > 1].count()) # 2번 이상 나온 중복 이름 몇 개인지 확인
feature_dup_df[feature_dup_df['column_index'] > 1].head() # 뭐 있는지 보기

column_index    42
dtype: int64


,column_index
column_name,
"fBodyAcc-bandsEnergy()-1,16",3
"fBodyAcc-bandsEnergy()-1,24",3
"fBodyAcc-bandsEnergy()-1,8",3
"fBodyAcc-bandsEnergy()-17,24",3
"fBodyAcc-bandsEnergy()-17,32",3


In [ ]:
## 중복된 피처명에 _1, _2를 부여해서 새로운 피처명을 부여해서 새로운 피처명을 가지는 데이터 프레임 반환

def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(), # cumcount 누적해서 갯수 세기, 이름이 3번 나온다면, 첫 번째는 0, 두 번째는 1, 세 번째는 2라는 번호를 순서대로 매기기
                                  columns=['dup_cnt']) # 매긴 번호를 새로운 열에 저장
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer') # reset_index 기존 인덱스를 일반 데이터로 변환, merge로 이름 옆에 중복 번호를 함께 표기
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name',
                                                              'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1]) # 원래 이름 뒤에 _2를 붙이기
                                                                if x[1] > 0 else x[0], axis=1) # 중복 번호가 0보다 크면 -> 두번째부터는 번호를 부여(_2, _3 ..) x[0]이면 그대로, axis=1 작업을 가로줄 단위로 반복하기
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1) # 작업을 가로줄 단위로 반복하기
    return new_feature_name_df